
# SLEIPNIR - Analysis

# Sequential Light Echo Identification Protocol Nearly In Realtime
# Primary analysis script

**What this notebook does**, for every spectrum listed in the sample file:

1. Reads the spectrum (applying the Galactic extinction and redshift corrections where the file says they are still needed)
2. Applies the redshift cuts from `Initial_Cuts.config`
3. Cleans / smooths it as set in `Spectral_Processing.config` (skylines, rebinning, FFT, median, Gaussian, Savitzky-Golay, telluric masking)
4. Measures every line in `Line_Lists.lines_for_analysis()` (continuum fit, pEQW, line flux, SNRs)
5. Scores the lines in `Line_Lists.lines_for_scoring()` using the thresholds in `ECLE_Scoring.config`, sets the candidate flag, line ratios, AGN flag and Lick indices
6. Writes one row per accepted spectrum to the results file and (optionally) the processed spectrum

**Inputs** - a tab separated sample file with the columns: `Target, Spectrum_Path, SpecType, MJD, RA, DEC, Redshift, Redshift_Err, Redshift_Corrected_Flag, Extinction, Extinction_Corrected_Flag, Flux_Scaled_Flag, Wavelength_System`, (only `SpecType = ASCII` is currently supported).
Missing `Redshift` / `Extinction` values are filled in (z = 0 and treated as already corrected / SFD map x 0.86).
`Wavelength_System` is an optional column (AIR / VACUUM) where a per-spectrum wavelength system can be declared.

**Note** All measurements, the saved processed spectra and output line names use the AIR wavelength scale, though initiall processing is conducted on the initial system of the spectra.

**Sentinel values** used in the results file and the line dictionaries:

| Value | Meaning |
|---|---|
| `-999` | Line (or its continuum windows) not covered by the spectrum |
| `-9999` | Measurement was not finite (e.g. a gap in the spectrum) |
| `-1000` | Line ratio / Lick index could not be determined (absorption, zero flux, missing line) |
| `-500` | Lick index calculation failed with an exception |

**Line dictionary slots** (each entry of `Spectral_Lines_For_Analysis` is a list - see also `Line_Lists.lines_for_analysis`):
`[0]` rest wavelength, `[1]` plot colour, `[2]` shift of the whole spectrum, `[3]` shift in the analysed region, `[4]` wavelength region, `[5]` flux region, `[6]` fitted continuum, `[7]` continuum scaled flux, `[8]` pEQW,
`[9]` feature region mean, `[10]` peak region max, `[11]` scoring result, `[16]` line flux, `[17]` relative max SNR, `[20]` strong (high SNR) flag, `[21]` error region, `[22]` per pixel SNR region,
`[23]` relative min SNR, `[24]` error spectrum max SNR, `[25]` error spectrum min SNR, `[26]` continuum standard deviation. Slots `[12]-[15]`, `[18]`, `[19]` are currently unused / reserved for survey specific parameters.

**Companion notebook:** `SLEIPNIR_Plots` repeats the line measurements and scoring on the saved processed spectra, so any change to the measurement / scoring code here must be mirrored there.


In [ ]:
#############################################################################################################
# User Inputs
#############################################################################################################

Sample_Name = 'Example_Data'
Sample_Meta_File = 'Example_InputFile.ascii'
# Only used if the input file has no 'Wavelength_System' column - per spectrum values ('AIR' / 'VACUUM') in the file take priority
Default_Wavelength_System = 'Air'

In [ ]:
#############################################################################################################
# Imports and initial set up
#############################################################################################################

import time
import os
import traceback

import pandas as pd
import numpy as np
import scipy.constants as constants

from tqdm.notebook import tqdm
from pylick.indices import IndexLibrary
from colorama import Fore, Style
from dustmaps.sfd import SFDQuery
from datetime import datetime
from astropy.coordinates import SkyCoord
from astropy.table.pprint import conf
from scipy.ndimage import binary_dilation

from astropy import units as u

conf.max_lines = -1
conf.max_width = -1

np.seterr(invalid='raise')

# SLEIPNIR specific module files
import Modules.Core.Core_Functions as Core_Functions
import Modules.Plotting.Plotting_Functions as Plotting_Functions
import Modules.Core.Config_Readers as Config_Readers
import Modules.Spectra.Spectra_Functions as Spectra_Functions
import Modules.Spectra.Line_Lists as Line_Lists

c = constants.value('speed of light in vacuum') / 1000  # Speed of light in kms^-1

# Required for internal extinction value determination
sfd = SFDQuery()

# Lick indices set up
index_list = np.array([24, 26, 32]) # Lick indices to be calculated - Numbers reference table.dat in pyLick
index_names = IndexLibrary(index_keys=index_list).names # Loading the desired indices

# The Lick values used below are looked up by name, so check the table numbers above still select them
assert {'D4000', 'Hdelta_A', 'Hgamma_A'} <= set(index_names), f'Unexpected Lick indices selected: {index_names}'

#######################
# Configuration
#######################
# The vast majority of the code's configuration is handled by three separate configuration files:
# A cuts configuration file: Details the requirements for candidate flagging and which spectra are selected in the first place
# A processing configuration file: Deals with how the spectra are smoothed etc.
# A scoring configuration file: Details the requirements for each individual line to score

# Each file has internal documentation explaining each parameter
# These are pathed to via the following variables:

Cuts_Configuration_File = "Configuration_Files/Cuts/Initial_Cuts.config"
Processing_Configuration_File = "Configuration_Files/Processing/Spectral_Processing.config"
Scoring_Configuration_File = "Configuration_Files/Scoring/ECLE_Scoring.config"
Plots_Configuration_File = "Configuration_Files/Plots/Plots_Config.config"

#######################
# Flags
#######################

consistency_check = False # If True prints a subsection of the data arrays of each spectrum at every processing step
verbose = False # If True some functions / analysis steps will provide intermediate outputs
quiet = False # If True only minimal outputs are displayed
debug = False
save_all_processed_spectra = True # If false, only the processed spectra of SLEIPNIR selected candidates are saved

# Marks a line as 'not usable for this spectrum' using the sentinel values the rest of the pipeline expects:
#   -999  = line (or its continuum windows) not covered by the spectrum
# Used both when the line lies outside the spectrum and when the spectrum ends inside the continuum windows.
# NB: the dictionary entries persist between objects, so every slot the plotting / results code reads is reset here.
def Mark_Line_Uncovered(line_entry):
	for slot in (2, 3, 4, 5, 6, 7, 9, 10, 11, 21, 22, 23, 24, 25, 26):
		line_entry[slot] = np.nan
	for slot in (8, 16, 17, 20):
		line_entry[slot] = -999

def vacuum_to_air(wave_vac):
	"""Vacuum -> air wavelengths (Angstrom), IAU / Morton (2000) standard refractivity formula. Plain floats / arrays in and out."""
	wave_vac = np.asarray(wave_vac, dtype=float)
	s2 = (1e4 / wave_vac) ** 2
	n = 1 + 0.0000834254 + 0.02406147 / (130 - s2) + 0.00015998 / (38.9 - s2)
	return wave_vac / n

def air_to_vacuum(wave_air):
	"""Air -> vacuum (Angstrom), same refractivity formula evaluated at the air wavelength (error << 0.001 A)."""
	wave_air = np.asarray(wave_air, dtype=float)
	s2 = (1e4 / wave_air) ** 2
	n = 1 + 0.0000834254 + 0.02406147 / (130 - s2) + 0.00015998 / (38.9 - s2)
	return wave_air * n

# Sanity check against NIST pairs from Line_Lists (tolerance covers NIST rounding)
for _vac, _air in ((6564.60, 6562.79), (5008.240, 5006.843), (3728.48, 3727.42)):
	assert abs(vacuum_to_air(_vac) - _air) < 0.03, f'vacuum_to_air({_vac}) = {vacuum_to_air(_vac):.3f}, expected ~{_air}'


In [ ]:
#######################
# Configuration Processing
# 
# Handles the configuration files
# All of these functions will terminate the script if a configuration error is detected
#######################

# Cuts Configuration
Configured_Cuts = Config_Readers.cuts_configuration_reader(filepath=Cuts_Configuration_File)

Low_z_Cut = Configured_Cuts[0]
High_z_Cut = Configured_Cuts[1]

# Processing Configuration
Configured_Processing_Steps = Config_Readers.processing_configuration_reader(filepath=Processing_Configuration_File)

Spectres_Rebin = Configured_Processing_Steps[0]
Rebin_Res = Configured_Processing_Steps[1]

FFT_Denoise = Configured_Processing_Steps[2]
FFT_Threshold = Configured_Processing_Steps[3]

Gaussian_Smooth = Configured_Processing_Steps[4]
Gaussian_Sigma = Configured_Processing_Steps[5]

Median_Filter = Configured_Processing_Steps[6]
Median_Filter_Kernel = Configured_Processing_Steps[7]

Sav_Gol = Configured_Processing_Steps[8]
Sav_Gol_Window = Configured_Processing_Steps[9]
Sav_Gol_Poly = Configured_Processing_Steps[10]

Problematic_Skyline_Removal = Configured_Processing_Steps[11]
Harsh_Skyline_Removal = Configured_Processing_Steps[12]
Strong_Sky_Post_Rebin_Harsh_Removal = Configured_Processing_Steps[13]
Telluric_Band_Masking = Configured_Processing_Steps[14]

# Scoring Configuration

Configured_Scoring = Config_Readers.scoring_configuration_reader(filepath=Scoring_Configuration_File)

Candidate_Threshold = Configured_Scoring[0]
Candidate_Score_Max = Configured_Scoring[1]

# Only used in the (currently unsupported) wavelength mode - the lower / upper wave region config values do nothing in 'Shift' mode
Lower_Wave = Configured_Scoring[2]
Upper_Wave = Configured_Scoring[3]

Lower_Shift = Configured_Scoring[4]
Upper_Shift = Configured_Scoring[5]

Line_Region = Configured_Scoring[6]
Feature_Region = Configured_Scoring[7]

LineDetection_pEQW_Threshold = Configured_Scoring[8]
LineDetection_Peak_Threshold = Configured_Scoring[9]
Line_Peak_Region_Min_Threshold = Configured_Scoring[10]

Relative_SNR_Threshold = Configured_Scoring[11]
Min_Relative_SNR_Threshold = Configured_Scoring[12]
Min_Espec_SNR_Threshold = Configured_Scoring[13]
Strong_SNR_Threshold = Configured_Scoring[14]
Weak_FeVII_SNR_Threshold = Configured_Scoring[15]

# Plots Configuration

Configured_Plots = Config_Readers.plotting_configuration_reader(filepath=Plots_Configuration_File)
Co_Ord_Summary_Plot = Configured_Plots

# Wavelength or Shift
Mode = 'Shift' # Only 'Shift' is currently valid in this initial release

startT = time.time()

########################
# Line 'list' Dictionaries
########################

# Every spectrum is converted to air as the last processing step (see vacuum_to_air), so only the air lists are used
Spectral_Lines_For_Analysis = Line_Lists.lines_for_analysis_air() # All the lines here are analysed
Spectral_Lines_For_Scoring = Line_Lists.lines_for_scoring_air() # All the lines here factor into the ECLE scoring
Spectral_Lines_For_Analysis_Names = list(Spectral_Lines_For_Analysis.keys()) # Display ready line labels

########################
# Output Directories Setup
########################

Timestamp = str(datetime.today().date())

# Configuration and generation of output directory save paths
Main_SavePath = os.path.join(f"Outputs/{Sample_Name}_{Timestamp}")
Results_SavePath = os.path.join(Main_SavePath, f"Result_Files")
Processed_Spectra_SavePath = os.path.join(Results_SavePath, f"Processed_Spectra")

Core_Functions.directory_creation(Main_SavePath)
Core_Functions.directory_creation(Results_SavePath)
Core_Functions.directory_creation(Processed_Spectra_SavePath)

print('Initial configuration complete')

Exception_Values = [-1000, -999, -995, -9999, 0]
# Values that indicate where lines are not accessible etc. - Currently only used to exclude values from the line ratio calculation
# (see the sentinel table at the top of the notebook; -995 is not currently assigned anywhere)

# This config value encodes the specific set up used for each run in the results file to make this traceable.
# NB: it covers the rebinning / smoothing settings and the redshift cuts only - the skyline and telluric switches and all of the
# scoring thresholds are NOT part of it. SLEIPNIR_Plots must build the identical string (it is compared against this one).
Settings_Config = f'{int(Spectres_Rebin)}_{Rebin_Res:g}_{int(FFT_Denoise)}_{int(FFT_Threshold)}_{int(Gaussian_Smooth)}_{Gaussian_Sigma}_{int(Median_Filter)}_{Median_Filter_Kernel}_{int(Sav_Gol)}_{Sav_Gol_Window}_{Sav_Gol_Poly}_{Low_z_Cut}_{High_z_Cut}'

In [ ]:
Target_Information = pd.read_csv(Sample_Meta_File, comment='#', sep='\t')

Required_Columns = [
	'Target', 'Spectrum_Path', 'SpecType', 'MJD', 'RA', 'DEC', 'Redshift', 'Redshift_Err', 'Redshift_Corrected_Flag',
	'Extinction', 'Extinction_Corrected_Flag', 'Flux_Scaled_Flag'
]
Missing_Columns = [column for column in Required_Columns if column not in Target_Information.columns]
if Missing_Columns:
	raise KeyError(f'The input file {Sample_Meta_File} is missing the column(s): {Missing_Columns}')

Invalid_Flux_Flags = Target_Information['Target'][~Target_Information['Flux_Scaled_Flag'].isin([0, 1])]
if len(Invalid_Flux_Flags) > 0:
	raise ValueError(f'Flux_Scaled_Flag must be exactly 0 or 1 for every spectrum - check: {Invalid_Flux_Flags.tolist()}')

if 'Wavelength_System' not in Target_Information.columns:
	print(f"{Fore.YELLOW}WARNING:{Style.RESET_ALL} No 'Wavelength_System' column in {Sample_Meta_File} - assuming '{Default_Wavelength_System}' for every spectrum")
	Target_Information['Wavelength_System'] = Default_Wavelength_System

Target_Information['Wavelength_System'] = Target_Information['Wavelength_System'].astype(str).str.strip().str.upper()
Invalid_Systems = Target_Information['Target'][~Target_Information['Wavelength_System'].isin(['AIR', 'VACUUM'])]
if len(Invalid_Systems) > 0:
	raise ValueError(f"Wavelength_System must be 'AIR' or 'VACUUM' for every spectrum - check: {Invalid_Systems.tolist()}")

Targets = Target_Information['Target']
Spectrum_Paths = Target_Information['Spectrum_Path']
Spectrum_Types = Target_Information['SpecType']

MJDs = Target_Information['MJD']

RAs = Target_Information['RA']
DECs = Target_Information['DEC']

Redshifts = Target_Information['Redshift']
Redshift_Errs = Target_Information['Redshift_Err']
Redshift_Corrected_Flags = Target_Information['Redshift_Corrected_Flag']

Extinctions = Target_Information['Extinction']
Extinction_Corrected_Flags = Target_Information['Extinction_Corrected_Flag']

Flux_Scaled_Flags = Target_Information['Flux_Scaled_Flag']

Wavelength_Systems = Target_Information['Wavelength_System']

del Target_Information

In [ ]:
Results_Storage_File = open(f"{Results_SavePath}/SLEIPNIR_Results_File_for_{Sample_Name}.ascii","w")
Results_Storage_File.write(
	"ID\tSpectrum_Number\tSpectrum_Path\t"
	"MJD\tRA\tDEC\t"
	"Redshift\tRedshift_Err\tRedshift_Corrected_Flag\t"
	"EBV\tExtinction_Corrected_Flag\t"
)

for ll, line in enumerate(Spectral_Lines_For_Analysis_Names):

	Results_Storage_File.write(
		f"{Spectral_Lines_For_Analysis_Names[ll]}_pEQW\t"
		f"{Spectral_Lines_For_Analysis_Names[ll]}_Line_Flux\t"
		f"{Spectral_Lines_For_Analysis_Names[ll]}_SNR\t"
		f"{Spectral_Lines_For_Analysis_Names[ll]}_High_SNR_Flag\t"
	)

Results_Storage_File.write(
	"NII_over_H_ALPHA_Ratio\tOIII_over_H_BETA_Ratio\tSII_over_H_ALPHA_Ratio\tOI6300_over_H_ALPHA_Ratio\tOIII5007_over_OIII4959\tHeII_over_Halpha\t"
    "D4000\tD4000_Err\tLick_HGamma_Index\tLick_HGamma_Index_Err\tLick_HDelta_Index\tLick_HDelta_Index_Err\t"
    "Detected_FeVII_Flag\tDetected_FeX_Flag\tDetected_FeXI_Flag\tDetected_FeXIV_Flag\t"
    "Detected_FeVII_3759_Flag\tDetected_FeVII_5159_Flag\tDetected_FeVII_5721_Flag\t"
    "Strong_FeVII_Flag\tStrong_FeX_Flag\tStrong_FeXI_Flag\tStrong_FeXIV_Flag\t"
    "Strong_FeVII_3759_Flag\tStrong_FeVII_5159_Flag\tStrong_FeVII_5721_Flag\t"
    "Quiescent_Flag\tBalmer_Strong_Flag\tBalmer_Moderately_Strong_Flag\t"
    "Score\tSettings_Config\tCandidate_Flag\tLocal_Score_Threshold\tThreshold_Modifier\tThreshold_Modifier_Condition\tAGN_Flag\t"
    )

Results_Storage_File.write(
	"Wavelength_System\n"
)

if Co_Ord_Summary_Plot: # Currently only one plot type is used in this path - prevents unneeded folder generation if that plot is not desired
	Summary_Plots_SavePath = os.path.join(Main_SavePath, f"Summary_Plots")
	Core_Functions.directory_creation(Summary_Plots_SavePath)

	Plotting_Functions.plot_sky(
		RAs,
		DECs,
		values=np.linspace(0,1,len(RAs)),
		labels=Targets,
        projection='mollweide',
        title=f'{Sample_Name} - Object Skymap',
		save_path=f'{Summary_Plots_SavePath}/{Sample_Name}_Skymap.pdf'

	)

# Sample Composition Files
Sample_Composition_Path = os.path.join(Main_SavePath, f"Sample_Composition_Files")
Core_Functions.directory_creation(Sample_Composition_Path)
Sample_Composition_Data_File = open(f"{Sample_Composition_Path}/Sample_Composition_SummaryFile_{Sample_Name}.ascii", 'w')

Accepted_Targets = []
Rejected_Targets = []

Spectrum_Number = 0
Successful_Objects_Count = 0
Failed_Objects_Count = 0
Candidate_Count = 0
ObjectCount = 0
Strong_FeVII_Object_Count = 0
Strong_FeX_Object_Count = 0
Strong_FeXI_Object_Count = 0
Strong_FeXIV_Object_Count = 0
Strong_FeVII_3759_Object_Count = 0
Strong_FeVII_5159_Object_Count = 0
Strong_FeVII_5721_Object_Count = 0

Failed_NII_over_Halpha_Line_Ratio = []
Failed_OIII_over_Hbeta_Line_Ratio = []
Failed_SII_over_Halpha_Line_Ratio = []
Failed_OI_over_Halpha_Line_Ratio = []
Failed_OIII5007_over_OIII4959_Ratio = []
Failed_HeII_over_Halpha_Ratio = []
Failed_Lick = []

for xx, target in tqdm(enumerate(Targets),total=len(Targets)):

	Target = Targets[xx]
	Spectrum_Path = Spectrum_Paths[xx]
	Spectrum_Type = Spectrum_Types[xx].upper()

	MJD = MJDs[xx]

	RA = RAs[xx]
	DEC = DECs[xx]
	Co_Ords = SkyCoord(RAs[xx], DECs[xx], unit='deg')

	Redshift = Redshifts[xx]
	Redshift_Err = Redshift_Errs[xx]
	Redshift_Corrected_Flag = Redshift_Corrected_Flags[xx]

	Extinction = Extinctions[xx]
	Extinction_Corrected_Flag = Extinction_Corrected_Flags[xx]

	Flux_Scaled_Flag = Flux_Scaled_Flags[xx]

	Wavelength_System = Wavelength_Systems[xx]

	if debug:

		print(Target)
		print(Spectrum_Path)
		print(Spectrum_Type)
		print(MJD)
		print(RA)
		print(DEC)
		print(Redshift)
		print(Redshift_Err)
		print(Redshift_Corrected_Flag)
		print(Extinction)
		print(Extinction_Corrected_Flag)

	# If a value for extinction is not provided, it is automatically determined from the object's co-ordinates
	if not np.isfinite(Extinction):
		print(f'{Fore.YELLOW}WARNING:{Style.RESET_ALL} Extinction has not been provided for {Target} - Determining automatically from given co-ordinates')
		EBV_SFD = sfd(Co_Ords) # Value expected for photometry
		EBV_SF11 = EBV_SFD * 0.86 # Value expected for spectra
		Extinction = EBV_SF11

	# Fallback for redshift not being provided
	if not np.isfinite(Redshift):
		print(f"{Fore.YELLOW}WARNING:{Style.RESET_ALL} Redshift has not been provided for {Target} - Assuming z = 0 / spectrum has previously been redshift corrected")
		Redshift = 0
		Redshift_Err = 0
		Redshift_Corrected_Flag = 1

	# Fallback for redshift uncertainty not being provided
	# NB: Redshift uncertainty is not-currently considered by SLEIPNIR but is retained for ease of sample bookkeeping
	if not np.isfinite(Redshift_Err):
		print(f"{Fore.YELLOW}WARNING:{Style.RESET_ALL} Redshift uncertainty has not been provided for {Target} - Assuming 0")
		Redshift_Err = 0

	if Spectrum_Type == 'ASCII':

		Spectrum_Data = Spectra_Functions.ascii_spectrum_reader(
			filepath = Spectrum_Path,
			z = Redshift,
			extinction = Extinction,
			z_corrected_flag = Redshift_Corrected_Flag,
			extinction_corrected_flag = Extinction_Corrected_Flag,
			multiplication_flag = int(Flux_Scaled_Flag == 0),

		)

		Wave = Spectrum_Data[0]
		Flux = Spectrum_Data[1]
		Flux_Err = Spectrum_Data[2]

		cleaned_flux = Flux.copy()
		cleaned_flux[cleaned_flux==0] = np.nan

		# Mask every negative pixel plus Neg_Pad pixels either side (symmetric, and safe at both ends of the spectrum)
		Neg_Pad = 1
		with np.errstate(invalid='ignore'): # nan < 0 is simply False, but np.seterr(invalid='raise') is set globally
			negative_pixels = np.asarray(getattr(cleaned_flux, 'value', cleaned_flux), dtype=float) < 0
		negative_mask = binary_dilation(negative_pixels, structure=np.ones(2 * Neg_Pad + 1, dtype=bool))
		cleaned_flux[negative_mask] = np.nan

		cleaned_flux_i = np.arange(len(cleaned_flux))
		nan_mask = np.isfinite(cleaned_flux)
		Processed_Flux = np.interp(cleaned_flux_i, cleaned_flux_i[nan_mask], cleaned_flux[nan_mask])

		cleaned_ivar = Flux_Err.copy()
		cleaned_ivar[cleaned_ivar==0] = np.nan

		Flux = Processed_Flux
		Inverse_Variance = cleaned_ivar
		Flux_Err = 1 / np.sqrt(Inverse_Variance)

	else:
		raise ValueError(f"{Target}: unsupported SpecType '{Spectrum_Type}' - only ASCII is currently implemented")

	if consistency_check:
		print('######\nData post readin: ')
		print(Wave[0:20], type(Wave))
		print(Flux[0:20], type(Flux))
		print(Flux_Err[0:20], type(Flux_Err))

	if Redshift_Corrected_Flag == 0:
		low_zcut = (Redshift > Low_z_Cut)
		# To better filter stars and redshift errors
		# This Cut only applied to spectra that have not been previously (or are assumed to have been) redshift corrected
	else:
		low_zcut = True

	high_zcut = (Redshift < High_z_Cut)
	# At higher redshifts, more of the desired spectral region is redshifted beyond the observable range reducing reliability

	select = low_zcut & high_zcut

	if select:
		Accepted_Targets.append(Target)

		Candidate_Score = 0
		Strong_FeVII_Flag = 0
		Strong_FeX_Flag = 0
		Strong_FeXI_Flag = 0
		Strong_FeXIV_Flag = 0
		Strong_FeVII_3759_Flag = 0
		Strong_FeVII_5159_Flag = 0
		Strong_FeVII_5721_Flag = 0

		Detected_FeVII_Flag = 0
		Detected_FeX_Flag = 0
		Detected_FeXI_Flag = 0
		Detected_FeXIV_Flag = 0
		Detected_FeVII_3759_Flag = 0
		Detected_FeVII_5159_Flag = 0
		Detected_FeVII_5721_Flag = 0

		Secondary_FeVII_Line_Flag = 0

		#################################
		# Skyline Interpolation
		#################################

		# Skyline lists (observed frame, rest frame). Needed by the harsh / strong removal steps below as well, so these are
		# defined whether or not the problematic skyline interpolation is active

		if Wavelength_System.upper() == 'AIR':
			
			problematic_sky_list, problematic_sky_list_z_corrected = Line_Lists.problematic_skylines_air(z=Redshift, mode='DESI')
			all_sky_list, all_sky_list_z_corrected = Line_Lists.sky_lines_air(z=Redshift)
			
		elif Wavelength_System.upper() == 'VACUUM':
			problematic_sky_list, problematic_sky_list_z_corrected = Line_Lists.problematic_skylines_vacuum(z=Redshift, mode='DESI')
			all_sky_list, all_sky_list_z_corrected = Line_Lists.sky_lines_vacuum(z=Redshift)

		if Problematic_Skyline_Removal:

			if Spectrum_Number == 0:
				print('Problematic skyline interpolation is active')

			half_width = 8 # Angstrom either side of each (rest frame) skyline that is interpolated over

			flux_in = np.array(Flux, dtype=float)  # copy, native byte order
			sky_mask = np.zeros(len(flux_in), dtype=bool)
			for centre in problematic_sky_list_z_corrected:
				sky_mask |= np.abs(Wave.value - centre) < half_width
			flux_in[sky_mask] = np.nan

			good = np.isfinite(flux_in)
			flux_in = np.interp(np.arange(len(flux_in)), np.flatnonzero(good), flux_in[good])

			# The interpolated flux has to be written back - previously it was calculated and then discarded
			Flux = flux_in * Flux.unit

			if consistency_check:
				print('######\nData post skyline processing: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		###########
		# Rebinning
		###########

		if Spectres_Rebin:

			if Spectrum_Number == 0 and not quiet:
				print('Rebinning Active')

			spectres_data = Spectra_Functions.spectres_rebin(
				flux=Flux.value,
				wave=Wave.value,
				error=Flux_Err.value,
				desired_res=Rebin_Res
			)

			Flux = spectres_data[0] * u.Unit('erg cm-2 s-1 AA-1')
			Wave = spectres_data[1] * u.Unit('AA')
			Flux_Err = spectres_data[2] * u.Unit('erg cm-2 s-1 AA-1')

			if consistency_check:
				print('######\nData post rebinning: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		if Harsh_Skyline_Removal:

			if Spectrum_Number == 0 and not quiet:
				print('Harsh skyline interpolation is active')

			for ss, line in enumerate(all_sky_list):

				difference = np.absolute(Wave.value - all_sky_list_z_corrected[ss])

				purge = np.argsort(difference)[:2]
				Flux[purge] = np.nan

			if consistency_check:
				print('######\nData post Harsh skyline processing: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		if Strong_Sky_Post_Rebin_Harsh_Removal:

			if Spectrum_Number == 0 and not quiet:
				print('Post rebin strong skyline removal is active')

			for ss, line in enumerate(problematic_sky_list_z_corrected):

				difference = np.absolute(Wave.value - problematic_sky_list_z_corrected[ss])

				purge = np.argsort(difference)[:6]
				Flux[purge] = np.nan

			if consistency_check:
				print('######\nData post strong skyline processing: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		cleaned_flux = Flux.copy()
		cleaned_flux[cleaned_flux==0] = np.nan

		cleaned_flux_i = np.arange(len(cleaned_flux))
		nan_mask = np.isfinite(cleaned_flux)
		Processed_Flux = np.interp(cleaned_flux_i, cleaned_flux_i[nan_mask], cleaned_flux[nan_mask])

		cleaned_Error = Flux_Err.copy()
		cleaned_Error[cleaned_Error==0] = np.nan

		Flux = Processed_Flux
		Error = cleaned_Error

		if consistency_check:
			print('######\nData post cleaning: ')
			print(Wave[0:20], type(Wave))
			print(Flux[0:20], type(Flux))
			print(Flux_Err[0:20], type(Flux_Err))

		#################################
		# Smoothing
		#################################

		if FFT_Denoise:

			FFT_Deactivation = 0 # Will flip to 1 if nans are detected (e.g., after pixel flagging)
			FFT_Nan_Interp = True # Will interpolate the flux axis is a nan is detected prior to running the FFT

			if Spectrum_Number == 0:
				print('FFT Active')

			if any(np.isnan(Flux)):

				if not quiet or verbose:
					print('Fast Fourier Transform Alert: Nan detected')

				if FFT_Nan_Interp:

					if not quiet or verbose:
						print('Interpolation Allowed: Interpolating across the nans')

					flux_i = np.arange(len(Flux.value))
					nan_mask = np.isfinite(Flux.value)
					interpolated_flux = np.interp(flux_i, flux_i[nan_mask], Flux.value[nan_mask])

					filtered_flux = Spectra_Functions.fft_filter_signal(interpolated_flux, threshold=FFT_Threshold)
					Flux = filtered_flux * u.Unit('erg cm-2 s-1 AA-1')

				else:
					if not quiet or verbose:
						print('Interpolation not allowed: Moving on without running the FFT')
					FFT_Deactivation = 1

			else:
				filtered_flux = Spectra_Functions.fft_filter_signal(
					Flux.value, threshold=FFT_Threshold
				)
				Flux = filtered_flux * u.Unit('erg cm-2 s-1 AA-1')

			if consistency_check:
				print('######\nData post FFT: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		if Median_Filter:

			if Spectrum_Number == 0:
				print('Median Filter Active')

			# Runs a median filter, defaults to a kernel of size 3 but can be set via the function call
			med_filtered_flux = Spectra_Functions.spectral_median_filter(
				Flux.value, med_filter_kernel= Median_Filter_Kernel
			)
			Flux = med_filtered_flux * u.Unit('erg cm-2 s-1 AA-1')

			if consistency_check:
				print('######\nData post Median filtering: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		if Gaussian_Smooth:

			if Spectrum_Number == 0:
				print('Gaussian Smoothing Active')

			gaussian_smoothed_flux = Spectra_Functions.gaussian_smoothing(
				Flux.value, sigma_value = Gaussian_Sigma
			)
			Flux = gaussian_smoothed_flux * u.Unit('erg cm-2 s-1 AA-1')
			gaussian_smoothed_error = Spectra_Functions.gaussian_smoothing(Error, sigma_value=Gaussian_Sigma)
			Error = gaussian_smoothed_error

			if consistency_check:
				print('######\nData post Gaussian smoothing: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		if Sav_Gol:

			if Spectrum_Number == 0:
				print('Sav Gol Filter Active')

			sav_gol_smoothed_flux = Spectra_Functions.sav_gol_filter(
				Flux.value, sav_gol_window=Sav_Gol_Window, sav_gol_poly=Sav_Gol_Poly
			)
			Flux = sav_gol_smoothed_flux * u.Unit('erg cm-2 s-1 AA-1')

			if consistency_check:
				print('######\nData post Sav Gol filtering: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		#################################
		# Drastic removal of the telluric contamination regions
		#################################

		if Telluric_Band_Masking:

			Wave = Wave.value
			Flux = Flux.value

			Telluric_Edges = [[7790, 8105], [8280, 8665], [8760, 9005], [9310, 9570]]
			if Wavelength_System == 'VACUUM':
				Telluric_Edges = [air_to_vacuum(edge).tolist() for edge in Telluric_Edges]

			telluric_bands_z_corrected = [(np.array(edge) / (1 + Redshift)).tolist() for edge in Telluric_Edges]

			for ww, item in enumerate(Wave):

				if telluric_bands_z_corrected[0][0] < Wave[ww] < telluric_bands_z_corrected[0][1]:
					Flux[ww] = np.nan

				elif telluric_bands_z_corrected[1][0] < Wave[ww] < telluric_bands_z_corrected[1][1]:
					Flux[ww] = np.nan

				elif telluric_bands_z_corrected[2][0] < Wave[ww] < telluric_bands_z_corrected[2][1]:
					Flux[ww] = np.nan

				elif telluric_bands_z_corrected[3][0] < Wave[ww] < telluric_bands_z_corrected[3][1]:
					Flux[ww] = np.nan

				elif Wave[ww] > telluric_bands_z_corrected[3][1]:
					break

			Wave = Wave * u.AA
			Flux = Flux * u.Unit('erg cm-2 s-1 AA-1')  # Flux

			if consistency_check:
				print('######\nData post drastic telluric masking: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		#################################
		# Conversion to the air scale (last processing step - everything above ran in the spectrum's own system)
		#################################
		if Wavelength_System == 'VACUUM':
			Wave = vacuum_to_air(Wave.value) * u.AA

			if consistency_check:
				print('######\nData post wavelength scale conversion: ')
				print(Wave[0:20], type(Wave))
				print(Flux[0:20], type(Flux))
				print(Flux_Err[0:20], type(Flux_Err))

		#################################
		# Per Pixel SNR
		#################################

		Flux_Values = np.asarray(getattr(Flux, 'value', Flux), dtype=float)
		Err_Values = np.asarray(getattr(Error, 'value', Error), dtype=float)

		Per_Pixel_SNR = np.full_like(Flux_Values, np.nan)
		valid = np.isfinite(Flux_Values) & np.isfinite(Err_Values) & (Err_Values > 0)
		Per_Pixel_SNR[valid] = Flux_Values[valid] / Err_Values[valid]

		Mean_SNR = np.nanmean(Per_Pixel_SNR)

		##########################################################################################################
		# Spectral read-in + pre-processing for one spectrum completed
		##########################################################################################################

		#################################
		# Line pEQW measurements using a linear continuum fit (the flux is divided by the continuum, not subtracted)
		# NB: Lines are processed in the order of Line_Lists.lines_for_analysis(). [FeVII]6088 must come before the weaker
		# FeVII lines as they only score if Detected_FeVII_Flag has already been set.
		#################################

		for ii, item in enumerate(Spectral_Lines_For_Analysis):

			Line_Location = Spectral_Lines_For_Analysis[item][0]

			# Velocity shift from line
			Shift = (((np.array(Wave) * c) / Line_Location) - c) * (u.kilometer / u.second)

			# Cut down the region for analysis (Lower_Shift to Upper_Shift around the line, from the scoring config)
			# The line is treated as not covered if the spectrum ends more than 50 Angstroms short of it. Spectra that stop closer
			# than that are still analysed, so the continuum check further down also has to catch partial coverage

			if Wave[-1].value < Line_Location - 50 or Wave[1].value > Line_Location + 50:

				if verbose:
					print(
						f"Analysis region \n{Spectral_Lines_For_Analysis_Names[ii]} is not fully covered in the rest frame."
					)

				Mark_Line_Uncovered(Spectral_Lines_For_Analysis[item])

			else:
				Shift_Region, Wave_Region, Flux_Region, Error_Region, Per_Pixel_SNR_Region = Spectra_Functions.region_cutter(
					shift=Shift,
					wave=Wave,
					flux=Flux,
					error=Error,
					per_pixel_snr=Per_Pixel_SNR,
					low_cut=Lower_Shift,
					high_cut=Upper_Shift,
					mode=Mode
				)


				# Generate and remove a continuum fit from the flux data - currently linear other functions seem worse
				# The scaled flux output - Continuum[1] - is scaled to a continuum value of 1
				# The returned continuum values - Continuum[0] - is relative to the original unscaled spectrum
				Continuum = Spectra_Functions.continua_maker(
					xaxis=Shift_Region,
					flux=Flux_Region,
					line_name=list(Spectral_Lines_For_Analysis)[ii],
					line_loc=Line_Location,
					object_name=Target,
					mode=Mode
				)

				# Partial coverage: the spectrum ends inside a continuum window, so no continuum can be fitted. Treat the line as not
				# covered (as the scoring threshold modifier expects) rather than letting eqw_measurement fail on the empty continuum
				if len(Continuum[0]) == 0:
					Mark_Line_Uncovered(Spectral_Lines_For_Analysis[item])
					continue

				# Measure the resulting pEQW of the feature

				pEQW_Spec, Line_Flux, Relative_max_snr, Relative_min_snr, Espec_max_snr, Espec_min_snr, Continuum_STDEV = Spectra_Functions.eqw_measurement(
					flux=Flux_Region,
					true_continuum=Continuum[0],
					scaled_flux=Continuum[1],
					continuum_regions = Continuum[2],
					wave=Wave_Region,
					shift=Shift_Region,
					error=Error_Region,
					per_pixel_snr=Per_Pixel_SNR_Region,
					line_loc=Line_Location,
					peak_region_range=Line_Region,
					mode=Mode
				)

				# Store the results in the main dictionary

				Spectral_Lines_For_Analysis[item][2] = Shift
				Spectral_Lines_For_Analysis[item][3] = Shift_Region
				Spectral_Lines_For_Analysis[item][4] = Wave_Region
				Spectral_Lines_For_Analysis[item][5] = Flux_Region
				Spectral_Lines_For_Analysis[item][6] = Continuum[0]
				Spectral_Lines_For_Analysis[item][7] = Continuum[1]

				Spectral_Lines_For_Analysis[item][21] = Error_Region
				Spectral_Lines_For_Analysis[item][22] = Per_Pixel_SNR_Region

				if not np.isfinite(pEQW_Spec):
					Spectral_Lines_For_Analysis[item][8] = -9999
				else:
					Spectral_Lines_For_Analysis[item][8] = pEQW_Spec

				if not np.isfinite(Line_Flux):
					Spectral_Lines_For_Analysis[item][16] = -9999
				else:
					Spectral_Lines_For_Analysis[item][16] = Line_Flux

				if not np.isfinite(Relative_max_snr):
					Spectral_Lines_For_Analysis[item][17] = -9999
				else:
					Spectral_Lines_For_Analysis[item][17] = Relative_max_snr

				if not np.isfinite(Relative_min_snr):
					Spectral_Lines_For_Analysis[item][23] = -9999
				else:
					Spectral_Lines_For_Analysis[item][23] = Relative_min_snr

				if not np.isfinite(Espec_max_snr):
					Spectral_Lines_For_Analysis[item][24] = -9999
				else:
					Spectral_Lines_For_Analysis[item][24] = Espec_max_snr

				if not np.isfinite(Espec_min_snr):
					Spectral_Lines_For_Analysis[item][25] = -9999
				else:
					Spectral_Lines_For_Analysis[item][25] = Espec_min_snr

				if not np.isfinite(Continuum_STDEV):
					Spectral_Lines_For_Analysis[item][26] = -9999
				else:
					Spectral_Lines_For_Analysis[item][26] = Continuum_STDEV

				##############################
				# Candidate Conditions Checks
				##############################

				if Spectral_Lines_For_Analysis_Names[ii] in Spectral_Lines_For_Scoring:

					LineScore = Spectra_Functions.lin_con_candidate_line_identifier(
						line_name=Spectral_Lines_For_Analysis_Names[ii],
						line_peqw=Spectral_Lines_For_Analysis[item][8],
						peqw_threshold=LineDetection_pEQW_Threshold,
						continuum_removed_flux=Continuum[1],
						shift_points=Shift_Region,
						peak_threshold=LineDetection_Peak_Threshold,
						peak_region=Line_Region,
						feature_region=Feature_Region,
						peak_region_minima_threshold=Line_Peak_Region_Min_Threshold,
						relative_max_snr=Relative_max_snr,
						relative_min_snr=Relative_min_snr,
						espec_min_snr=Espec_min_snr,
						relative_snr_threshold=Relative_SNR_Threshold,
						min_relative_snr_threshold=Min_Relative_SNR_Threshold,
						min_espec_snr_threshold=Min_Espec_SNR_Threshold,
						strong_snr_threshold=Strong_SNR_Threshold,
						weak_FeVII_line_snr_threshold=Weak_FeVII_SNR_Threshold,
						main_fevii_flag=Detected_FeVII_Flag
					)

					Candidate_Score = Candidate_Score + LineScore[0]
					Spectral_Lines_For_Analysis[item][9] = LineScore[4]
					Spectral_Lines_For_Analysis[item][10] = LineScore[2]
					Spectral_Lines_For_Analysis[item][11] = LineScore[7]
					Spectral_Lines_For_Analysis[item][20] = LineScore[9]

					Secondary_FeVII_Line_Flag = Secondary_FeVII_Line_Flag + LineScore[10]

					if Spectral_Lines_For_Analysis_Names[ii] == '[FeXIV]5303' and Target == 'Fearow':
						print(f'{Target} FeXIV LineScore: {LineScore}')
						print(f'Strong_SNR_Threshold={Strong_SNR_Threshold}, Relative_SNR_Threshold={Relative_SNR_Threshold}, Min_Relative_SNR_Threshold={Min_Relative_SNR_Threshold}, Min_Espec_SNR_Threshold={Min_Espec_SNR_Threshold}')
						print(f'max_rel={Relative_max_snr}, min_rel={Relative_min_snr}, espec_min={Espec_min_snr}')

					"""
					Records the detection of the scoring lines
					"""

					if Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]6087':
						if LineScore[0] != 0:
							Detected_FeVII_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeX]6375':
						if LineScore[0] != 0:
							Detected_FeX_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeXI]7892':
						if LineScore[0] != 0:
							Detected_FeXI_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeXIV]5303':
						if LineScore[0] != 0:
							Detected_FeXIV_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]3759':
						if LineScore[0] != 0:
							Detected_FeVII_3759_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]5159':
						if LineScore[0] != 0:
							Detected_FeVII_5159_Flag = 1

					elif Spectral_Lines_For_Analysis_Names[ii] == '[FeVII]5721':
						if LineScore[0] != 0:
							Detected_FeVII_5721_Flag = 1

				else:
					Spectral_Lines_For_Analysis[item][9] = np.nan
					Spectral_Lines_For_Analysis[item][10] = np.nan
					Spectral_Lines_For_Analysis[item][11] = np.nan

					if Relative_max_snr >= Strong_SNR_Threshold:
						Spectral_Lines_For_Analysis[item][20] = 1
					else:
						Spectral_Lines_For_Analysis[item][20] = 0

		if Spectral_Lines_For_Analysis['[FeVII]6087'][20] == 1:
			Strong_FeVII_Flag = 1
			Strong_FeVII_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeX]6375'][20] == 1:
			Strong_FeX_Flag = 1
			Strong_FeX_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeXI]7892'][20] == 1:
			Strong_FeXI_Flag = 1
			Strong_FeXI_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeXIV]5303'][20] == 1:
			Strong_FeXIV_Flag = 1
			Strong_FeXIV_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeVII]3759'][20] == 1:
			Strong_FeVII_3759_Flag = 1
			Strong_FeVII_3759_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeVII]5159'][20] == 1:
			Strong_FeVII_5159_Flag = 1
			Strong_FeVII_5159_Object_Count += 1

		if Spectral_Lines_For_Analysis['[FeVII]5721'][20] == 1:
			Strong_FeVII_5721_Flag = 1
			Strong_FeVII_5721_Object_Count += 1

		if Spectral_Lines_For_Analysis['[NII]6583'][16] in Exception_Values or Spectral_Lines_For_Analysis['Halpha'][16] in Exception_Values:
			NII_over_Halpha_Ratio = -1000

		else:
			try:
				NII_over_Halpha_Ratio = np.log10(
					Spectral_Lines_For_Analysis['[NII]6583'][16] / Spectral_Lines_For_Analysis['Halpha'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of NII_over_Halpha_Ratio:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				NII_over_Halpha_Ratio = -1000
				Failed_NII_over_Halpha_Line_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of NII_over_Halpha_Ratio:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				NII_over_Halpha_Ratio = -1000
				Failed_NII_over_Halpha_Line_Ratio.append(Target)

		if Spectral_Lines_For_Analysis['[OIII]5007'][16] in Exception_Values or Spectral_Lines_For_Analysis['Hbeta'][16] in Exception_Values:
			OIII_over_Hbeta_Ratio = -1000
		else:
			try:
				OIII_over_Hbeta_Ratio = np.log10(
					Spectral_Lines_For_Analysis['[OIII]5007'][16] / Spectral_Lines_For_Analysis['Hbeta'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of OIII_over_Hbeta_Ratio:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				OIII_over_Hbeta_Ratio = -1000
				Failed_OIII_over_Hbeta_Line_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of OIII_over_Hbeta_Ratio:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				OIII_over_Hbeta_Ratio = -1000
				Failed_OIII_over_Hbeta_Line_Ratio.append(Target)

		if Spectral_Lines_For_Analysis['[SII]6717_6731'][16] in Exception_Values or Spectral_Lines_For_Analysis['Halpha'][16] in Exception_Values:
			SII_over_Halpha_Ratio = -1000

		else:
			try:
				SII_over_Halpha_Ratio = np.log10(
					Spectral_Lines_For_Analysis['[SII]6717_6731'][16] / Spectral_Lines_For_Analysis['Halpha'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of SII_over_Halpha_Ratio:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				SII_over_Halpha_Ratio = -1000
				Failed_SII_over_Halpha_Line_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of SII_over_Halpha_Ratio:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				SII_over_Halpha_Ratio = -1000
				Failed_SII_over_Halpha_Line_Ratio.append(Target)

		if Spectral_Lines_For_Analysis['[OI]6300'][16] in Exception_Values or Spectral_Lines_For_Analysis['Halpha'][16] in Exception_Values:
			OI_over_Halpha_Ratio = -1000

		else:
			try:
				OI_over_Halpha_Ratio = np.log10(
					Spectral_Lines_For_Analysis['[OI]6300'][16] / Spectral_Lines_For_Analysis['Halpha'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of OI_over_Halpha_Ratio:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				OI_over_Halpha_Ratio = -1000
				Failed_OI_over_Halpha_Line_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of OI_over_Halpha_Ratio:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				OI_over_Halpha_Ratio = -1000
				Failed_OI_over_Halpha_Line_Ratio.append(Target)

		if Spectral_Lines_For_Analysis['[OIII]5007'][16] in Exception_Values or Spectral_Lines_For_Analysis['[OIII]4959'][16] in Exception_Values:
			OIII5007_over_OIII4959_Ratio = -1000

		else:
			try:
				OIII5007_over_OIII4959_Ratio = np.log10(
					Spectral_Lines_For_Analysis['[OIII]5007'][16] / Spectral_Lines_For_Analysis['[OIII]4959'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of [OIII]5007_over_[OIII]4959_Ratio:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				OIII5007_over_OIII4959_Ratio = -1000
				Failed_OIII5007_over_OIII4959_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of [OIII]5007_over_[OIII]4959_Ratio:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				OIII5007_over_OIII4959_Ratio = -1000
				Failed_OIII5007_over_OIII4959_Ratio.append(Target)

		if Spectral_Lines_For_Analysis['HeII4686'][16] in Exception_Values or Spectral_Lines_For_Analysis['Halpha'][16] in Exception_Values:
			HeII_over_Halpha_Ratio = -1000

		else:
			try:
				HeII_over_Halpha_Ratio = np.log10(
					Spectral_Lines_For_Analysis['HeII4686'][16] / Spectral_Lines_For_Analysis['Halpha'][16])

			except FloatingPointError:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of HeII_over_Halpha:\n"
						"Likely cause is one or both lines measured in absorption : Setting value to -1000 and continuing."
					)

				HeII_over_Halpha_Ratio = -1000
				Failed_HeII_over_Halpha_Ratio.append(Target)

			except RuntimeWarning:
				if not quiet or verbose:
					print(
						"\nlog10 error in the determination of HeII_over_Halpha:\n"
						"Looks like a zero got in there somewhere : Setting value to -1000 and continuing."
					)

				HeII_over_Halpha_Ratio = -1000
				Failed_HeII_over_Halpha_Ratio.append(Target)

		# Final safety net: any ratio that is still infinite / NaN (e.g. a zero that slipped past the checks above) is set to -1000
		# NB: The previous version of these checks mixed up the HeII and NII ratios and tested the OIII5007 / OIII4959 ratio twice
		for ratio_name, ratio_value, failed_list in (
				('NII/Halpha', NII_over_Halpha_Ratio, Failed_NII_over_Halpha_Line_Ratio),
				('OIII/Hbeta', OIII_over_Hbeta_Ratio, Failed_OIII_over_Hbeta_Line_Ratio),
				('SII/Halpha', SII_over_Halpha_Ratio, Failed_SII_over_Halpha_Line_Ratio),
				('OI/Halpha', OI_over_Halpha_Ratio, Failed_OI_over_Halpha_Line_Ratio),
				('OIII5007/OIII4959', OIII5007_over_OIII4959_Ratio, Failed_OIII5007_over_OIII4959_Ratio),
				('HeII/Halpha', HeII_over_Halpha_Ratio, Failed_HeII_over_Halpha_Ratio)
		):
			if not np.isfinite(ratio_value):
				if not quiet or verbose:
					print(f"Looks like a cheeky zero still managed to squeak through, setting {ratio_name} to -1000 for {Target}")

				failed_list.append(Target)

				if ratio_name == 'NII/Halpha': NII_over_Halpha_Ratio = -1000
				elif ratio_name == 'OIII/Hbeta': OIII_over_Hbeta_Ratio = -1000
				elif ratio_name == 'SII/Halpha': SII_over_Halpha_Ratio = -1000
				elif ratio_name == 'OI/Halpha': OI_over_Halpha_Ratio = -1000
				elif ratio_name == 'OIII5007/OIII4959': OIII5007_over_OIII4959_Ratio = -1000
				else: HeII_over_Halpha_Ratio = -1000

		if verbose:
			print("\n")
			print(Target)
			print(NII_over_Halpha_Ratio)
			print(OIII_over_Hbeta_Ratio)
			print(SII_over_Halpha_Ratio)
			print(OI_over_Halpha_Ratio)
			print(OIII5007_over_OIII4959_Ratio)
			print(HeII_over_Halpha_Ratio)
			print(Spectral_Lines_For_Analysis['[NII]6583'][16])
			print(Spectral_Lines_For_Analysis['Halpha'][16])
			print("\n")


		##############################
		# Lick Indices Calculation
		##############################

		try:
			Lick_Index_Data = Spectra_Functions.lick_index_calculation(Target, Wave, Flux, Error, index_list, index_names=index_names)

		except Exception:
			traceback.print_exc()
			if not quiet or verbose:
				print("Lick Index Calculation Failure")
			Lick_Index_Data = dict.fromkeys(index_names, (-500, -500))
			Failed_Lick.append(Target)

		for index in index_names:
			Lick_Index_Data[index] = (-1000, -1000) if not any(np.isfinite(Lick_Index_Data[index])) else Lick_Index_Data[index]

		D4000, D4000_Err = Lick_Index_Data["D4000"][0], Lick_Index_Data["D4000"][1]
		Lick_HDelta_Index, Lick_HDelta_Index_Err = Lick_Index_Data["Hdelta_A"][0], Lick_Index_Data["Hdelta_A"][1]
		Lick_HGamma_Index, Lick_HGamma_Index_Err = Lick_Index_Data["Hgamma_A"][0], Lick_Index_Data["Hgamma_A"][1]

		##############################
		# Additional Object Flagging
		##############################

		if Spectral_Lines_For_Analysis['Halpha'][8] > -3: # Flags on the strength of weak Balmer emission
			Quiescent_Flag = 1

		else:
			Quiescent_Flag = 0

		# -1000: no finite value, -500: the calculation failed - neither can be used for the Balmer flags
		if Lick_HDelta_Index in (-1000, -500):
			Balmer_Strong_Flag = -1
			Balmer_Moderately_Strong_Flag = -1

		else:
			if Lick_HDelta_Index - Lick_HDelta_Index_Err > 4:
				Balmer_Strong_Flag = 1
			else:
				Balmer_Strong_Flag = 0

			if Lick_HDelta_Index - Lick_HDelta_Index_Err > 1.31:
				Balmer_Moderately_Strong_Flag = 1
			else:
				Balmer_Moderately_Strong_Flag = 0

		#######
		# Candidate Filtering + Flagging
		#######

		# Checking for non-covered red lines with corresponding score threshold reduction

		if Spectral_Lines_For_Analysis['[FeXI]7892'][8] == -999:
			Candidate_Threshold_Modifier = - 2
			Candidate_Threshold_Modifier_Condition = 1

		else:
			Candidate_Threshold_Modifier = 0
			Candidate_Threshold_Modifier_Condition = 0

		Local_Candidate_Threshold = Candidate_Threshold + Candidate_Threshold_Modifier

		Candidate_Flag = 0
		AGN_Flag = 0

		if Candidate_Score >= Local_Candidate_Threshold and (Strong_FeVII_Flag+Strong_FeX_Flag+Strong_FeXI_Flag+Strong_FeXIV_Flag+Strong_FeVII_3759_Flag+Strong_FeVII_5159_Flag+Strong_FeVII_5721_Flag) == 0:
			Candidate_Flag = 1
			Candidate_Count += 1

		elif Candidate_Score >= Local_Candidate_Threshold and (Strong_FeVII_Flag+Strong_FeX_Flag+Strong_FeXI_Flag+Strong_FeXIV_Flag+Strong_FeVII_3759_Flag+Strong_FeVII_5159_Flag+Strong_FeVII_5721_Flag)> 0:
			Candidate_Flag = 2
			Candidate_Count += 1

		elif Candidate_Score < Local_Candidate_Threshold and (Strong_FeVII_Flag+Strong_FeX_Flag+Strong_FeXI_Flag+Strong_FeXIV_Flag+Strong_FeVII_3759_Flag+Strong_FeVII_5159_Flag+Strong_FeVII_5721_Flag)> 0:
			Candidate_Flag = 5
			Candidate_Count += 1

		elif Secondary_FeVII_Line_Flag >= 2:
			Candidate_Flag = 7
			Candidate_Count += 1

		if OIII_over_Hbeta_Ratio > ((0.61 / (NII_over_Halpha_Ratio - 0.05)) + 1.3) or NII_over_Halpha_Ratio >= 0.05:
			AGN_Flag += 1

		if OIII_over_Hbeta_Ratio >  ((0.72 / (SII_over_Halpha_Ratio - 0.32)) + 1.30) or SII_over_Halpha_Ratio >= 0.32:
			AGN_Flag += 1

		if OIII_over_Hbeta_Ratio >  ((0.73 / (OI_over_Halpha_Ratio + 0.59)) + 1.33) or OI_over_Halpha_Ratio >= -0.59:
			AGN_Flag += 1

		ObjectCount += 1
		Successful_Objects_Count += 1

		if Candidate_Flag !=0 or save_all_processed_spectra:

			# Saves the spectrum exactly as it was analysed - NB: the error column is 'Error' (zeros set to nan, smoothed if the
			# Gaussian smoothing is on), the version the SNRs were calculated from. Previously this wrote the unprocessed 'Flux_Err'
			pd.DataFrame({
				'Wavelength': np.asarray(getattr(Wave, 'value', Wave), dtype=float),
				'Flux': np.asarray(getattr(Flux, 'value', Flux), dtype=float),
				'FluxErr': np.asarray(getattr(Error, 'value', Error), dtype=float)
			}).to_csv(
				f'{Processed_Spectra_SavePath}/{Target}_{Spectrum_Number}_Processed_Spectrum.ascii', sep='\t', index=False, na_rep='nan'
			)

		Results_Storage_File.write(
			f"{Target}\t"
			f"{Spectrum_Number}\t"
			f"{Spectrum_Path}\t"
			f"{MJD}\t"
			f"{RA}\t"
			f"{DEC}\t"
			f"{Redshift}\t{Redshift_Err}\t{Redshift_Corrected_Flag}\t"
			f"{Extinction}\t{Extinction_Corrected_Flag}\t"
		)

		for ll, line in enumerate(Spectral_Lines_For_Analysis_Names):

			Results_Storage_File.write(
				f"{Spectral_Lines_For_Analysis[line][8]}\t"
				f"{Spectral_Lines_For_Analysis[line][16]}\t"
				f"{Spectral_Lines_For_Analysis[line][17]}\t"
				f"{Spectral_Lines_For_Analysis[line][20]}\t"
			)

		Results_Storage_File.write(
			f"{NII_over_Halpha_Ratio}\t{OIII_over_Hbeta_Ratio}\t{SII_over_Halpha_Ratio}\t"
			f"{OI_over_Halpha_Ratio}\t{OIII5007_over_OIII4959_Ratio}\t{HeII_over_Halpha_Ratio}\t"
			f"{D4000}\t{D4000_Err}\t{Lick_HGamma_Index}\t{Lick_HGamma_Index_Err}\t{Lick_HDelta_Index}\t{Lick_HDelta_Index_Err}\t"
			f"{Detected_FeVII_Flag}\t{Detected_FeX_Flag}\t{Detected_FeXI_Flag}\t{Detected_FeXIV_Flag}\t"
			f"{Detected_FeVII_3759_Flag}\t{Detected_FeVII_5159_Flag}\t{Detected_FeVII_5721_Flag}\t"
			f"{Strong_FeVII_Flag}\t{Strong_FeX_Flag}\t{Strong_FeXI_Flag}\t{Strong_FeXIV_Flag}\t"
			f"{Strong_FeVII_3759_Flag}\t{Strong_FeVII_5159_Flag}\t{Strong_FeVII_5721_Flag}\t"
			f"{Quiescent_Flag}\t{Balmer_Strong_Flag}\t{Balmer_Moderately_Strong_Flag}\t{Candidate_Score}\t"
			f"{Settings_Config}\t{Candidate_Flag}\t{Local_Candidate_Threshold}\t"
			f"{Candidate_Threshold_Modifier}\t{Candidate_Threshold_Modifier_Condition}\t{AGN_Flag}\t{Wavelength_System}\n"
		)
		Results_Storage_File.flush() # So a crash part way through a large sample does not lose the rows already written

		# Variable iteration
		Spectrum_Number += 1


	else:
		print(f'{Target}: Rejected')
		Rejected_Targets.append(Target)

		continue

Total_Targets = len(Targets)
Total_Accepted_Targets = len(Accepted_Targets)
Total_Rejected_Targets = len(Rejected_Targets)

# Closed first so the results are safely written even if something below fails
Results_Storage_File.close()

# Guarded so that an empty sample, or one where every spectrum is rejected, does not end in a ZeroDivisionError
Acceptance_Percentage = (Total_Accepted_Targets/Total_Targets) * 100 if Total_Targets > 0 else 0.0
Candidate_Percentage = (Candidate_Count/Total_Accepted_Targets) * 100 if Total_Accepted_Targets > 0 else 0.0

print(f"\n\nTotal number of spectra processed: {Total_Targets}")
print(f"Total number of spectra meeting examination cuts: {Total_Accepted_Targets} : {Acceptance_Percentage:.2f}% ")
print(f"No. of ECLE Candidates identified: {Candidate_Count} : {Candidate_Percentage:.2f}% ")

Sample_Composition_Data_File.write(f"\n\nTotal number of spectra processed: {Total_Targets}\n")
Sample_Composition_Data_File.write(f"Total number of spectra meeting examination cuts: {Total_Accepted_Targets}\n")
Sample_Composition_Data_File.write(f"No. of ECLE Candidates identified: {Candidate_Count}\n")

Sample_Composition_Data_File.close()

endT = time.time()
executionT = endT - startT

print("Done")
print(f"Script execution time: {executionT:.2f}s")